# Dive.ai 진행사항

> 벡터DB.ipynb / 파이프라인.ipynb 기준으로 작성된 개발 진행 현황 및 주간 일정입니다.


## 지난주 작업 (~ 2026-04-27)

### 핵심 기능 1 — 시나리오 & 캐릭터 빌더

| 항목 | 파일 | 상태 |
|------|------|------|
| ChromaDB 벡터 DB 구축 | 벡터DB.ipynb §1~5 | ✅ 완료 |
| 기승전결 스테이지 매핑 | 벡터DB.ipynb §6 | ✅ 완료 |
| RAG 검색 + 프롬프트 빌더 | 벡터DB.ipynb §7~8 | ✅ 완료 |
| 시나리오 생성 함수 | 벡터DB.ipynb §10 Cell 28 | ✅ 완료 |
| 캐릭터 생성 함수 | 벡터DB.ipynb §12 Cell 43 | ✅ 완료 |
| 빌더 통합 파이프라인 | 벡터DB.ipynb §12 Cell 44 | ✅ 완료 |

- 시나리오 생성 모델: gpt-4o-mini(전체적인 테스팅 시), gpt-4o(복잡한 프롬프트 테스팅, 시연 시), 다른 모델도 고려 후 추가
- 캐릭터 생성: AI캐릭터 + 유저캐릭터 + 주조연 2~4명 (시나리오 기반 자동 설계)

### 핵심 기능 2 — 시나리오 트랜스포머

| 항목 | 파일 | 상태 |
|------|------|------|
| 시나리오 트랜스포머 v1 (ScenarioTree) | 벡터DB.ipynb §11 Cell 30~32 | ✅ 완료 |
| 부분 재생성 함수 | 벡터DB.ipynb §11 Cell 35 | ✅ 완료 |
| GameState (호감도·선택이력·이벤트플래그) | 벡터DB.ipynb §11 Cell 37 | ✅ 완료 |
| 엔딩 조건 설계 (LLM 자동 생성) | 벡터DB.ipynb §11 Cell 38 | ✅ 완료 |
| 엔딩 판정 로직 (evaluate_ending) | 벡터DB.ipynb §11 Cell 39 | ✅ 완료 |

- 분기 구조: 기 → 승 → 전(공통) → [선택 A/B] → 전_A/전_B → 결_A/결_B - 수정 예정
- 이탈 감지 시 현재 노드 이후만 부분 재생성 (대화 요약·관계 상태 반영)


## 이번주 일정 (2026-04-28 ~ 2026-05-03)

> 목표: **웹사이트에 ui, 기능 구현, 인터랙티브 챗 시스템 프로토타입 완성**<br>
> 시나리오 트랜스포머 먼저 수정


### 트랙 1 — 시나리오 트랜스포머 수정

> **방향성 확정 — 동적 분기 + LLM 트리거 생성**

현재 구현(v1)은 완료 상태이나 설계 방향이 바뀌어 전면 재설계 진행.

**확정된 재설계 방향:**
- 고정 분기 구조(기→승→전[A/B]→결) → **분기점 동적 감지 + 동적 생성**
- 객관식 선택지 → **유저 자유 대사 입력 + 힌트 카드 UI**
- LLM이 나침반 요약본 생성 시 **trigger_conditions 시나리오 특화 생성** → 매 턴 `trigger_branch` 플래그 반환
- **엔딩 분기점(결 단계 진입)은 고정**, 나머지 분기점은 동적 감지

| 항목 | 현재 상태 | 이번주 |
|------|----------|--------|
| ScenarioTree 분기 구조 (기→승→전→[A/B]→결) | ✅ v1 구현 완료 | 동적 분기 구조로 재설계 |
| 부분 재생성 함수 (`regenerate_from_node`) | ✅ v1 구현 완료 | 동적 생성 흐름에 맞게 수정 |
| 엔딩 조건 설계 (`GameState` + 호감도/플래그) | ✅ v1 구현 완료 | trigger_branch 연동으로 수정 |
| `generate_compass_summary()` | ❌ 미구현 | 나침반 요약본 + trigger_conditions JSON 생성 함수 신규 작성 |
| 동적 분기 생성 함수 | ❌ 미구현 | 유저 입력 해석 → 나침반+로어북 기반 다음 단계 동적 생성 |
| 결 단계 진입 감지 + evaluate_ending() 연동 | ❌ 미구현 | trigger_branch 감지 → evaluate_ending() 호출 |



### 트랙 2 — 인터랙티브 챗 시스템 프로토타입

> 참고 파일: `벡터DB.ipynb`, `파이프라인.ipynb`

파이프라인.ipynb 로드맵 기준 **1단계 마무리 + 3단계 핵심 구현** 목표.

#### 1단계 마무리 — 로어북 자동 초기화

- 시나리오 생성 완료 시 LLM으로 핵심 항목(지명 / 인물 관계 / 고유 명사 / 복선 디테일) 자동 추출
- 추출 항목 임베딩 → ChromaDB `lorebook` 컬렉션 색인
- `generate_scenario_full()` 완료 직후 자동 호출되도록 연동

```
시나리오 생성 완료
    → LLM 핵심 항목 추출
    → 임베딩 → ChromaDB lorebook 컬렉션 색인
```

#### 1단계 마무리 — `generate_compass_summary()` 신규 작성

- 기승전결 원문 → 나침반 요약본 (300~500 토큰) 생성
- 시나리오 특화 **trigger_conditions 3~5개** 함께 생성 (LLM이 장르·소재·캐릭터 맥락에서 직접 생성)
- 반환 형식: JSON (`compass_summary` + `trigger_conditions` 배열)

```python
# 반환 예시
{
    "compass_summary": "핵심 갈등 / 복선 목록 / 서사 목표 / 엔딩 방향 ...",
    "trigger_conditions": [
        "두 캐릭터 사이의 신뢰가 처음으로 균열을 일으키는 순간",
        "유저의 말이 캐릭터의 숨겨진 상처를 건드리는 순간",
        "과거의 복선이 현재 대화에서 회수되는 순간"
    ]
}
```

#### 2단계 마무리 — `generate_ending_conditions` 검증

- 실제 API 호출 테스트 및 `ending_condition` JSON 구조 검증
- `evaluate_ending` 통합 테스트 (해피·배드·트루 엔딩 전 케이스)
- LLM 출력 파싱 실패 시 fallback 처리 추가

#### 3단계 — 챗 시스템 핵심 구현

**시스템 프롬프트 조합 함수** (`build_system_prompt`)

| 레이어 | 내용 |
|--------|------|
| 고정 레이어 | 유저 페르소나 + 유저 노트 + 현재 ScenarioTree 노드 정보 + trigger_conditions |
| 동적 레이어 A | 로어북 Semantic Search → 문맥 유사 항목 실시간 주입 |
| 동적 레이어 B | 현재 기승전결 단계 기반 RAG 씬 컨텍스트 |
| 동적 레이어 C | 요약 기억 (10~15턴 초과 시 자동 요약 + 관계 상태 추출) |

**분기점 처리 — trigger_branch 패턴**

| 항목 | 내용 |
|------|------|
| 분기점 감지 | 매 턴 AI 응답의 `trigger_branch: true` 감지 |
| 힌트 카드 트리거 | trigger_branch: true → 힌트 카드 표시 → 유저 자유 대사 입력 |
| 유저 입력 해석 | AI가 대사 의도·방향 해석 → affinity_delta + 이벤트 플래그 기록 |
| 다음 단계 생성 | 나침반 요약본 + 로어북 + 대화 요약 기반 동적 생성 |
| 엔딩 분기점 | 결 단계 진입 조건 충족 시 → evaluate_ending() 호출 |
| 이탈 감지 | N턴 연속 시나리오 방향 이탈 → `regenerate_from_node()` 호출 |

```
매 턴 종료 시 체크
    ├─► trigger_branch: true? → 힌트 카드 → 유저 입력 → 동적 분기 생성
    ├─► 결 단계 진입?         → evaluate_ending() → 화면 7
    ├─► 이탈 감지?            → regenerate_from_node() 호출
    └─► 그 외                 → 다음 턴 자유 대화 계속
```



### 트랙 3 — 웹사이트 UI & 기능 구현 (화면 1~5)

> 참고 파일: `ui기능구현.ipynb`
> 이번주 범위: 화면 1 ~ 화면 5

#### 화면 1 — 진입: 콘텐츠 유형 & 장르 선택

- 콘텐츠 유형 선택 버튼: 만화 / 시리즈 / 영화 / 소설 / 고전
- 유형별 장르 목록 표시 + **무작위** 버튼
- 고전 선택 시 추가 UI: 국가 선택(한·중·일) → 해당 국가 장르 목록

#### 화면 2 — 소재 & 캐릭터 입력

- **섹션 A**: 소재 직접 입력 / AI에게 맡김 토글 + 자동 생성 소재 미리보기
- **섹션 B (AI 캐릭터)**: 이름·성격·외형·배경 각 입력창 우측 `AI` 뱃지 버튼
- **섹션 C (유저 캐릭터)**: 이름·성격·배경 각 입력창 우측 `AI` 뱃지 버튼
- 섹션 B/C 상단 안내 문구: `"AI 뱃지를 누르면 시나리오에 맞게 AI가 자동으로 설정해요."`
- 섹션별 **전체 AI에게 맡김** 버튼 제공
- 섹션 C 입력값 → 화면 5 페르소나 자동 연동

#### 화면 3 — 생성 중 로딩

- 단계별 진행 상태 텍스트 + 로딩 애니메이션
- 순서: 소재 분석 → 시나리오 작성 → 캐릭터 설계 → 인터랙티브 구조 변환 → 엔딩 조건 설계 → 완료

#### 화면 4 — 완료 화면: 시나리오 & 캐릭터 확인

- **섹션 A**: 기승전결 시나리오 요약 표시 (펼치기/접기, 단계별 탭)
- **섹션 B**: 등장인물 카드 (AI캐릭터 / 유저캐릭터 / 조연 2~4명) + 인라인 수정 가능
- AI 캐릭터 기준 이미지 생성 (이미지 API 미정 — 화면 구조만 구현)

#### 화면 5 — 플레이 시작 전 설정

- **유저 페르소나**: 화면 2 유저 캐릭터 정보 자동 채워짐, 자유 수정 가능
- **유저 노트**: AI가 항상 기억할 사항 자유 입력 (매 턴 시스템 프롬프트 고정 주입)
- **세션 옵션**: 출력 모델 / 추론 양 / 감성 / AI 주도 사건 / 사칭 설정 / 시작 설정
- **"대화 시작"** 버튼 → 화면 6 진입


## 지난주 작업 (2026-04-28 ~ 2026-05-05)

> 목표: 트랜스포머 재설계 + 인터랙티브 챗 시스템 + 백엔드 인프라 구축

### 트랙 1 — 시나리오 트랜스포머 재설계 (완료)

| 항목 | 파일 | 상태 |
|------|------|------|
| `GameStateV2` (기/승/전/결 단계·호감도·off_track·MIN_STAGE_TURNS 게이트) | chat_engine_v2.py | ✅ 완료 |
| `generate_compass()` — 나침반 요약본 + trigger_conditions + stage_constraints JSON | chat_engine_v2.py | ✅ 완료 |
| `generate_next_stage()` — 분기점 도달 시 유저 입력 반영, 다음 단계 동적 생성 | chat_engine_v2.py | ✅ 완료 |
| `generate_ending_scene()` — 결 단계 엔딩 씬 LLM 생성 | chat_engine_v2.py | ✅ 완료 |
| `generate_hint_card()` — 분기점 힌트 카드 문구 생성 | chat_engine_v2.py | ✅ 완료 |
| `regenerate_compass()` — off_track_count ≥ 3 시 나침반 부분 재생성 | chat_engine_v2.py | ✅ 완료 |

### 트랙 2 — 인터랙티브 챗 시스템 (완료)

| 항목 | 파일 | 상태 |
|------|------|------|
| `build_chat_system_prompt()` — 고정 레이어(나침반·trigger_conditions·단계제약·호감도기준) + 동적 레이어(로어북·요약·관계도) 조합 | chat_engine_v2.py | ✅ 완료 |
| `parse_chat_response()` — trigger_branch/trigger_ending/affinity_delta/off_track 파싱 + 백엔드 게이트 | chat_engine_v2.py | ✅ 완료 |
| `extract_lorebook_entries()` — 시나리오·나침반 기반 로어북 자동 초기화 | chat_engine_v2.py | ✅ 완료 |
| `get_lorebook_context()` — 최근 대화 문맥 기반 키워드 매칭 → 시스템 프롬프트 주입 | chat_engine_v2.py | ✅ 완료 |
| `update_summary_if_needed()` — 10턴마다 대화 요약 자동 갱신 | chat_engine_v2.py | ✅ 완료 |
| `generate_relationship_graph()` — 인물 관계도 자동 생성 (단계 전환 시 자동 업데이트) | chat_engine_v2.py | ✅ 완료 |
| SSE 스트리밍 채팅 `/chat/stream` — compass 유무 기준 v1 레거시 / v2 자동 라우팅 | main.py | ✅ 완료 |
| 사칭 모드(impersonation_enabled), 주사위 굴리기(dice_roll) | main.py, chat_engine_v2.py | ✅ 완료 |
| 대화 분기 기능 (`/topics/{id}/duplicate`) | main.py | ✅ 완료 |

### 트랙 3 — 백엔드 인프라 구축 (완료)

| 항목 | 파일 | 상태 |
|------|------|------|
| FastAPI CRUD API 완성 (유저·페르소나·토픽·메시지·요약·로어북·관계도) | main.py | ✅ 완료 |
| 빌더 파이프라인 SSE API `/builder/run` — 소재분석→시나리오→캐릭터→나침반→로어북→완료 6단계 스트리밍 | main.py | ✅ 완료 |
| 채팅 오프닝 메시지 자동 생성 API (`/topics/{id}/opening`) | main.py | ✅ 완료 |
| 토큰 사용량 추정 API (`/token-estimate/{id}`) | main.py | ✅ 완료 |
| DB 마이그레이션 — compass / game_state / lorebook_entries / relationship_graph 컬럼 추가 | models.py, main.py | ✅ 완료 |

### 팀원 테스팅 & 피드백 반영

지난주 완성된 프로토타입을 팀원들이 직접 테스팅하고 피드백을 공유함. 전달받은 피드백 사항을 모두 수정·반영 완료.

### 사용 모델 정리

| 역할 | 모델 | 방식 |
|------|------|------|
| **빌더 파이프라인** — 시나리오 생성, 캐릭터 설계, 나침반 생성, 로어북 초기화 | Vertex AI Gemini 3.1 Flash-Lite (`gemini-3.1-flash-lite-preview`) | Vertex AI (서비스 계정 인증) |
| **채팅 서사 엔진** — 단계 전환 생성, 힌트 카드, 엔딩 씬, 대화 요약, 인물 관계도, 나침반 재생성 | Vertex AI Gemini 3.1 Flash-Lite (`gemini-3.1-flash-lite-preview`) | Vertex AI (서비스 계정 인증) |
| **RAG 임베딩** — VectorDB 검색 쿼리 임베딩 | OpenAI `text-embedding-3-small` (1536차원) | OpenAI API |
| **채팅 모델** (유저 선택 1) | Gemini 3.1 Flash-Lite (`gemini-3.1-flash-lite-preview`) | Google AI Studio (무료) |
| **채팅 모델** (유저 선택 2) | Gemini 3.1 Flash-Lite (`gemini-3.1-flash-lite-preview-vertex`) | Vertex AI |
| **채팅 모델** (유저 선택 3) | Gemini 3 Flash (`gemini-3-flash-preview`) | Google AI Studio (무료) |
| **채팅 모델** (유저 선택 4) | Gemini 3 Flash (`gemini-3-flash-preview-vertex`) | Vertex AI |
| **채팅 모델** (유저 선택 5) | GPT-5.4 Nano (`gpt-5.4-nano`) | OpenAI API |


## 이번주 일정 (2026-05-06 ~ 2026-05-12)

> 목표: **미완성 UI 마무리 + 구글 로그인 + 이미지 API 연동 검토**


### 트랙 1— 세션 & 계정 관리

| 항목 | 내용 | 상태 |
|------|------|------|
| 구글 로그인 연동 | OAuth2 + 다중 user_id 지원 (현재 하드코딩 user_id=1) | 🔲 이번주 |
| 다중 세션 UI | + 버튼 → 새 대화창, 이름 수정/삭제 | ✅ 완료 (ChatInterface 좌측 사이드바) |
| 엔딩 아카이브 | 달성한 엔딩 목록 저장 + 조회 화면 | ⏳ 여건 보고 |

### 트랙 2 — 이미지 API 검토

| 항목 | 내용 | 상태 |
|------|------|------|
| 이미지 API 후보 비교 | nanobanana2 등 | 🔲 이번주 |
| AI 캐릭터 기준 이미지 생성 플로우 설계 | 얼굴 일관성 유지 방식 확정 | ⏳ API 확정 후 |

### 트랙 3 — 팀원 피드백 기반 추가 개선

지난주 테스팅에서 수집된 피드백 외에, 이번주 추가 테스팅을 통해 발견되는 부족한 부분 및 개선 요청 사항을 지속적으로 반영할 계획.

| 항목 | 내용 | 상태 |
|------|------|------|
| 추가 팀원 테스팅 | 이번주 구현 사항 포함 재테스팅, 신규 피드백 수집 | 🔲 이번주 |
| 피드백 기반 UI/UX 개선 | 테스팅 과정에서 발견된 불편 사항·버그 즉시 수정 | 🔲 이번주 |
| 피드백 기반 기능 추가 | 테스팅 결과 필요하다고 판단된 기능 우선순위 정리 후 반영 | 🔲 이번주 |

### 트랙 4 — 모델 업그레이드 & 테스팅

| 항목 | 내용 | 상태 |
|------|------|------|
| 빌더·서사 엔진 모델 업그레이드 | 현재 `gemini-3.1-flash-lite-preview` → `gemini-3.0-flash` 로 전환, 시나리오·나침반·엔딩·힌트카드 생성 품질 향상 | 🔲 이번주 |
| 채팅 상위 모델 추가 | 채팅 모델 선택지에 `gemini-3.0-flash`, `gemini-3.1-pro-preview` 등 상위 모델 추가 후 품질 비교 테스팅 | 🔲 이번주 |


## 전체 남은 작업 현황

| 단계 | 항목 | 상태 |
|------|------|------|
| 트랜스포머 재설계 | ScenarioTree 동적 분기 구조로 재설계 | ✅ 완료 |
| 트랜스포머 재설계 | `generate_compass()` 나침반 요약본 + trigger_conditions | ✅ 완료 |
| 트랜스포머 재설계 | 동적 분기 생성 함수 (`generate_next_stage`) | ✅ 완료 |
| 트랜스포머 재설계 | 결 단계 진입 감지 + `generate_ending_scene()` 연동 | ✅ 완료 |
| 1단계 빌더 마무리 | 로어북 자동 초기화 (`extract_lorebook_entries`) | ✅ 완료 |
| 3단계 챗 시스템 | 시스템 프롬프트 (고정+동적 레이어, trigger_conditions 포함) | ✅ 완료 |
| 3단계 챗 시스템 | 대화 요약기 + 관계 상태 추적 (`update_summary_if_needed`, `generate_relationship_graph`) | ✅ 완료 |
| 3단계 챗 시스템 | trigger_branch 처리 + 힌트 카드 + 동적 분기 생성 | ✅ 완료 |
| 5단계 로어북 | 유저 직접 관리 UI — 항목 추가·수정·삭제, 카테고리 분류(장소·인물·사건·복선) | ✅ 완료 |
| UI 화면 1~6 | 프론트엔드 UI 구현 (Screen0~5, ChatInterface) | ✅ 완료 |
| UI 화면 5 세션 옵션 | 출력 모델 선택·감성·사칭 모드 on/off (ChatInterface 우측 사이드바 SYSTEM 섹션) | ✅ 완료 |
| UI 화면 7 (잔여) | 엔딩 결과 전체 화면 — 달성 분기 경로, 재도전·공유 버튼 | 🔲 이번주 |
| 6단계 세션 | 구글 로그인 + 다중 user_id 지원 | 🔲 이번주 |
| 이미지 API | 후보 비교 및 확정, 이미지 자리 UI 선구현 | 🔲 이번주 |
| 모델 업그레이드 | 빌더·서사 엔진 모델 gemini-3.1-flash-lite → gemini-3.0-flash 업그레이드, 상위 채팅 모델 추가 | 🔲 이번주 |
| 4단계 엔딩 | 엔딩 이미지 생성 (얼굴 일관성 포함) | ⏳ API 확정 후 |
| 6단계 세션 | 엔딩 아카이브 | ⏳ 추후 |
